In [1]:
import astropy.units as u
from astropy.coordinates import SkyCoord

ModuleNotFoundError: No module named 'astropy'

In [ ]:
!pip -q install reproject imageio
from google.colab import drive
drive.mount('/content/drive')

import os, json, logging
import numpy as np, pandas as pd, imageio
from astropy.io import fits
from astropy.wcs import WCS
from astropy.time import Time
from reproject import reproject_interp
logging.getLogger('astropy').setLevel(logging.ERROR)

BASE = "/content/drive/MyDrive/spherex_cache"    # FITS descargados
SALIDA = "/content/drive/MyDrive/spherex_web"    # lo que irá al repo

def procesar(clave, detector="SPHEREx-D2", n=16, zoom=8):
    o = OBJETIVOS[clave]
    coord = posicion_actual(clave)
    size = o["size"]

    m = pd.read_csv(f"{BASE}/{clave}/manifest.csv")
    m = m[m.detector == detector]
    m = m[[os.path.exists(f"{BASE}/{clave}/{a}") for a in m.archivo]].sort_values("mjd")
    if len(m) == 0:
        print("Sin archivos para", clave, detector); return
    elegidos = np.unique(np.linspace(0, len(m) - 1, min(n, len(m))).astype(int))
    m = m.iloc[elegidos]

    escala = 6.15 / 3600
    npix = int(size / escala)
    destino = WCS(naxis=2)
    destino.wcs.ctype = ["RA---TAN", "DEC--TAN"]
    destino.wcs.crval = [coord.ra.deg, coord.dec.deg]
    destino.wcs.crpix = [npix / 2 + 0.5, npix / 2 + 0.5]
    destino.wcs.cdelt = [-escala, escala]

    d = detector.split("-")[-1]                   # "D2"
    os.makedirs(f"{SALIDA}/{clave}/frames", exist_ok=True)
    frames = []
    for i, fila in enumerate(m.itertuples()):
        try:
            with fits.open(f"{BASE}/{clave}/{fila.archivo}") as h:
                datos = h[1].data.astype(float)
                hdr = h[1].header
                w = WCS(hdr)
                try:                              # longitud de onda en la posición del objetivo
                    x, y = w.world_to_pixel(coord)
                    sw = WCS(hdr, fobj=h, key="W"); sw.sip = None
                    lam, _ = sw.pixel_to_world(x, y)
                    lam = round(float(lam.to(u.micrometer).value), 4)
                except Exception:
                    lam = None
        except Exception as e:
            print("omitido", fila.archivo, e); continue

        img, _ = reproject_interp((datos, w), destino, shape_out=(npix, npix))
        img -= np.nanmedian(img)
        lo, hi = np.nanpercentile(img, [5, 99.5])
        img = np.nan_to_num(np.clip((img - lo) / (hi - lo), 0, 1)) * 255
        img = np.kron(np.flipud(img), np.ones((zoom, zoom))).astype(np.uint8)

        nombre = f"frames/{d}_{i:02d}.png"
        imageio.imwrite(f"{SALIDA}/{clave}/{nombre}", img)
        frames.append(dict(url=nombre, mjd=round(fila.mjd, 3),
                           date=Time(fila.mjd, format="mjd").strftime("%Y-%m-%d"),
                           detector=d, wavelength_um=lam))

    manifest = dict(target=dict(name=clave, ra=round(coord.ra.deg, 5),
                                dec=round(coord.dec.deg, 5), fov=size),
                    frames=frames)
    with open(f"{SALIDA}/{clave}/manifest_{d}.json", "w") as f:
        json.dump(manifest, f, indent=1)
    print(clave, d, len(frames), "cuadros")

procesar("r_dra")